# Spatial Domains with BANKSY

Leiden clustering on a transcriptomic PCA assigns labels based solely on each cell's own expression vector: two cells receive the same label because their genes look alike, regardless of whether they occupy the same neighborhood or opposite ends of the tissue. This is appropriate for *cell typing*.

For *tissue domain* segmentation, a cell at a domain boundary should carry signal from its surroundings, not just its own transcriptome. [BANKSY](https://www.nature.com/articles/s41588-024-01664-3) (Singhal et al., *Nature Genetics* 2024) concatenates a neighborhood-averaged feature vector onto each cell's expression before PCA/clustering: spatial context is baked into the input rather than evaluated post-hoc.

This notebook builds the BANKSY pipeline step by step, then introduces `tl.banksy_sweep` as the shortcut for scanning multiple parameter combinations. Setup follows the same steps as every other notebook: load, QC/filter, normalize, log-transform.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

In [ ]:
mdata = ab.datasets.xenium()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)

## 1. The baseline: expression-only Leiden

The standard pipeline from notebook 01: PCA on HVG expression, transcriptomic KNN graph, Leiden. This result is preserved as `leiden` for comparison against BANKSY's spatial domains later.

In [ ]:
ab.pp.highly_variable_genes(mdata, n_top_genes=2000)
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)  # transcriptomic graph -> uns['neighbors']
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors', key_added='leiden')

mdata['xenium'].obs['leiden'].value_counts()

## 2. Building the BANKSY feature matrix

`pp.banksy_matrix` reads `obsp['{neighbors_key}_connectivities']` to identify each cell's spatial neighbors, so `tl.spatial_neighbors` must run first. It writes only `uns['spatial_neighbors']`, leaving the transcriptomic graph built above (`uns['neighbors']`) untouched, so the two coexist independently.

For each cell, `banksy_matrix` computes **M0** (mean neighbor expression; optionally **|M1|**, an azimuthal Gabor magnitude, via `agf=True`) then concatenates a √(1−λ)-scaled copy of the cell's own expression with a √λ-scaled copy of M0. Output is in `obsm['X_banksy']` with shape `(n_cells, 2 × n_genes)`.

λ controls the self-vs-neighborhood balance:
- **λ ≈ 0.2** — predominantly self-expression with mild neighborhood smoothing. Appropriate for cell typing.
- **λ ≈ 0.8** — predominantly neighborhood. Appropriate for spatial domain segmentation, where a cell's label should reflect its surrounding tissue context.

In [ ]:
ab.tl.spatial_neighbors(mdata, coord_type='delaunay', delaunay_max_dist=200)
ab.pp.banksy_matrix(mdata, lambda_param=0.8)

mdata['xenium'].obsm['X_banksy'].shape

## 3. PCA → graph → Leiden on the BANKSY matrix

The pipeline is identical to standard clustering — only the input representation changes:

1. `tl.pca(mdata, use_rep='X_banksy')` — PCA on the concatenated matrix (per the BANKSY paper: cluster the PCA embedding, not the raw concatenated features).
2. `tl.transcriptional_neighbors(mdata, use_rep='X_pca_banksy')` — KNN graph on the BANKSY-PCA coordinates. Spatial information is already embedded via M0; this builds an ordinary graph for Leiden.
3. `tl.cluster` on that graph, written to `key_added='banksy_leiden'` so it doesn't overwrite the baseline `leiden` column.

In [ ]:
ab.tl.pca(mdata, use_rep='X_banksy', key_added='X_pca_banksy')
ab.tl.transcriptional_neighbors(mdata, use_rep='X_pca_banksy')
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors', key_added='banksy_leiden')

mdata['xenium'].obs['banksy_leiden'].value_counts()

## 4. Expression clusters vs. spatial domains, side by side

`pl.points(...)` returns a lazy `SpatialPanel`; `|` composes two panels side by side without rendering either first. Left: `leiden` (expression only). Right: `banksy_leiden` (λ=0.8, spatially smoothed) — same tissue, same cells.

`banksy_leiden` regions should appear more contiguous — less salt-and-pepper intermixing — since each cell's label now incorporates its neighbors' expression.

In [ ]:
panel = ab.pl.points(mdata, color='leiden')  # not rendered yet
(panel | ab.pl.points(mdata, color='banksy_leiden')).show()

BANKSY (λ=0.8, res=0.5) identifies more domains than the expression-only Leiden baseline above -- 13 versus 10 here, a difference of 3. The additional domains typically reflect boundary zones that expression-only clustering merged: cells sitting at the interface between two distinct tissue regions often have an intermediate expression profile (mixing transcripts from adjacent cell types) that pure expression clustering places in one cluster or the other, while BANKSY's neighborhood term pulls them toward whichever domain their surroundings actually belong to, sometimes splitting them off as their own boundary domain. In the side-by-side tissue plot, BANKSY domains should appear with more spatially contiguous, less "salt-and-pepper" boundaries than the expression-only map -- individual cells are less likely to be assigned a different label than their immediate neighbors, since the neighborhood term anchors them to local context.

`+` overlays layers on shared axes: useful here to check whether domain boundaries align with actual cell boundaries rather than cutting through them arbitrarily.

In [ ]:
(ab.pl.points(mdata, color='banksy_leiden') + ab.pl.boundaries(mdata, alpha=0.3)).show()

The overlay reveals whether BANKSY domain boundaries align with physical cell boundaries or cut through them. Well-aligned transitions (where the color change in `pl.points()` coincides with a gap between adjacent cell boundary polygons) confirm that BANKSY is assigning whole cells to domains. Isolated islands of a minority color surrounded by cells of a different domain label indicate genuine boundary cells: their own expression and their neighborhood expression partially disagree, so BANKSY places them at the margin. Systematic misalignment, where a domain-color transition runs across a dense block of cells that all share similar boundary shapes, suggests the Leiden resolution is too high and is splitting a biologically uniform region into artificial sub-domains; reduce resolution or λ.

## 5. `tl.banksy_sweep`: the same pipeline, grid-searched

λ and resolution are rarely known in advance — λ trades off cell-type purity against spatial smoothness; resolution controls domain count. `tl.banksy_sweep` runs the complete sequence (`banksy_matrix` → PCA → `sc.pp.neighbors` → Leiden) for each `(λ, resolution)` combination and stores each result as a named `obs` column.

A 2×2 grid illustrates the API; a production scan typically covers a wider range.

In [ ]:
summary = ab.tl.banksy_sweep(
    mdata,
    lambda_params=[0.2, 0.8],
    resolutions=[0.3, 0.5],
)
summary

The sweep table above lets each factor's effect be measured directly: compare cluster count moving along a row (λ fixed, resolution changing) against moving down a column (resolution fixed, λ changing) -- computing both deltas fairly, at the same fixed value of the other parameter, rather than comparing the largest available contrast for one factor against a smaller contrast for the other. In this run, resolution moves the cluster count by more than λ does at every fixed value of the other parameter -- worth checking explicitly rather than assuming λ dominates just because it's BANKSY's namesake parameter. That doesn't make λ unimportant: cluster *count* and cluster *quality* (whether domains reflect real spatial coherence, checked with `tl.spatial_silhouette` rather than assumed from count alone) are different questions, and λ's effect shows up mainly in the latter -- how strongly each cell's label is pulled toward its neighborhood consensus, not how many discrete groups the partition lands on. For a production sweep, the "stable plateau" is the parameter region where increasing resolution no longer splits existing domains but instead creates new, spatially coherent ones; if it only adds very small clusters with no clear spatial or marker-gene signal, the domains are over-segmented.

Each row of the summary is one `(λ, resolution)` run, named `banksy_sweep_L{lambda}_R{resolution}` — that obs column holds its cluster labels, and `uns['banksy_sweep']` holds this table for later reference. Below, the low-λ (cell-typing-flavored) and high-λ (domain-flavored) runs at the same resolution are compared directly on the tissue.

In [ ]:
low_lambda_key = summary.loc[(summary['lambda'] == 0.2) & (summary['resolution'] == 0.3), 'obs_key'].item()
high_lambda_key = summary.loc[(summary['lambda'] == 0.8) & (summary['resolution'] == 0.3), 'obs_key'].item()

(ab.pl.points(mdata, color=low_lambda_key) | ab.pl.points(mdata, color=high_lambda_key)).show()

Comparing low-λ (0.2, left) and high-λ (0.8, right) at the same resolution shows what the neighborhood term adds. The low-λ map resembles expression-only Leiden: clusters follow transcriptomic identity and may appear salt-and-pepper where adjacent cells have different expression profiles. The high-λ map shows larger, more spatially contiguous patches: cells that differ slightly in expression from their neighbors are pulled toward the neighborhood consensus, reducing isolated mis-assigned cells. If the two maps look nearly identical for a given dataset, the neighborhood signal is weak: either the tissue lacks spatially structured domains and expression-only clustering is the appropriate choice, or `delaunay_max_dist` is too permissive and is blending spatially distant cells into each neighborhood average.

## Manual pipeline vs. `tl.banksy_sweep`

Build the pipeline by hand (sections 2–4) to understand exactly what BANKSY changes about the input, or when a single `(λ, resolution)` is already known. Use `tl.banksy_sweep` to scan multiple combinations and compare domain counts in `uns['banksy_sweep']`.